# Data Quality — Setup Notebook

**Run this notebook once** to create the tables that power the validation system.

- `check_registry` — stores the model names, DAX expressions, and row-level fail thresholds you want to validate
- `check_baseline_config` — stores whether each check compares by MODEL baseline row or STATIC numeric value
- `validation_results` — stores the results of each run, including the fail threshold used for each row

After this setup, go to `data_quality_add_checks_notebook` to add your first checks.

In [ ]:
# -- Step 1: Configuration -----------------------------------------------------
# Preferred: load shared settings from Fabric config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
except Exception:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")

LEGACY_DEFAULT_FAIL_DELTA_PCT_THRESHOLD = 0.01

# --- check_registry: the table business users fill in ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.check_registry (
    check_name                 STRING  NOT NULL COMMENT 'Same name across all models so the job knows to compare them',
    model_name                 STRING  NOT NULL COMMENT 'Friendly label - display name only',
    workspace_id               STRING  NOT NULL COMMENT 'Fabric workspace GUID for stable identity',
    dataset_id                 STRING  NOT NULL COMMENT 'Semantic model GUID for stable identity',
    workspace_name             STRING  NOT NULL COMMENT 'Fabric workspace display name',
    dataset_name               STRING  NOT NULL COMMENT 'Semantic model / dataset display name',
    dax_expression             STRING  NOT NULL COMMENT 'DAX returning a single number, e.g. EVALUATE ROW("value", [Sales Amount])',
    run_frequency              STRING  NOT NULL DEFAULT 'ONCE_PER_DAY' COMMENT 'ONCE_PER_DAY (run max once per day) or MULTIPLE_PER_DAY (can run multiple times)',
    fail_delta_pct_threshold   DOUBLE  NOT NULL DEFAULT 0.01 COMMENT 'Percent delta threshold for this row. Example: 0.5 means fail when absolute delta exceeds 0.5%',
    is_active                  BOOLEAN NOT NULL DEFAULT true COMMENT 'Set to false to skip this row without deleting it',
    is_baseline                BOOLEAN NOT NULL DEFAULT false COMMENT 'True for exactly one row per check_name - used as the comparison reference'
 )
USING DELTA
COMMENT 'Add one row per model per check - all rows sharing a check_name will be compared. PRIMARY KEY uses stable IDs.'
""")

# --- check_baseline_config: one row per check_name to choose MODEL vs STATIC baseline mode ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.check_baseline_config (
    check_name             STRING  NOT NULL COMMENT 'Check identifier shared across models',
    baseline_mode          STRING  NOT NULL DEFAULT 'MODEL' COMMENT 'MODEL uses is_baseline row, STATIC uses static_baseline_value',
    static_baseline_value  DOUBLE  COMMENT 'Required when baseline_mode = STATIC',
    updated_at             TIMESTAMP NOT NULL DEFAULT current_timestamp()
 )
USING DELTA
COMMENT 'Per-check baseline mode configuration. MODEL uses check_registry.is_baseline; STATIC uses static number.'
""")

# Ensure legacy registries include all columns used by runtime notebooks.
for add_col_sql in [
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (workspace_id STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (dataset_id STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (workspace_name STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (dataset_name STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (is_active BOOLEAN)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (run_frequency STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (fail_delta_pct_threshold DOUBLE)",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ADD COLUMNS (is_baseline BOOLEAN)",
]:
    try:
        spark.sql(add_col_sql)
        print(f"Applied schema update: {add_col_sql}")
    except Exception as e:
        if "already exists" not in str(e).lower():
            raise RuntimeError(
                f"Failed schema update for check_registry: {add_col_sql} -> {str(e)[:300]}"
            )

# Normalize legacy null/blank values after backfill so runtime filters remain stable.
spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET is_active = true
WHERE is_active IS NULL
""")

spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET run_frequency = 'ONCE_PER_DAY'
WHERE run_frequency IS NULL OR TRIM(run_frequency) = ''
""")

spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET run_frequency = UPPER(TRIM(run_frequency))
WHERE run_frequency IS NOT NULL
  AND UPPER(TRIM(run_frequency)) IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
""")

spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET fail_delta_pct_threshold = {LEGACY_DEFAULT_FAIL_DELTA_PCT_THRESHOLD}
WHERE fail_delta_pct_threshold IS NULL
""")

spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET is_baseline = false
WHERE is_baseline IS NULL
""")

# Ensure constraint also exists for already-created tables.
try:
    spark.sql(f"""
    ALTER TABLE {FULL_SCHEMA}.check_registry
    ADD CONSTRAINT pk_check_registry PRIMARY KEY (check_name, workspace_id, dataset_id)
    """)
    print("PRIMARY KEY constraint added to existing check_registry table")
except Exception as e:
    msg = str(e).lower()
    if "already exists" in msg:
        print("PRIMARY KEY constraint already present")
    else:
        raise RuntimeError(
            "Failed to enforce PRIMARY KEY on check_registry. "
            "Concurrent-writer operation requires enforced uniqueness on "
            "(check_name, workspace_id, dataset_id). "
            f"Original error: {str(e)[:300]}"
        )

# Ensure baseline config has a strict one-row-per-check invariant.
try:
    spark.sql(f"""
    ALTER TABLE {FULL_SCHEMA}.check_baseline_config
    ADD CONSTRAINT pk_check_baseline_config PRIMARY KEY (check_name)
    """)
    print("PRIMARY KEY constraint added to check_baseline_config")
except Exception as e:
    msg = str(e).lower()
    if "already exists" in msg:
        print("PRIMARY KEY constraint already present on check_baseline_config")
    else:
        raise RuntimeError(
            "Failed to enforce PRIMARY KEY on check_baseline_config (check_name). "
            f"Original error: {str(e)[:300]}"
        )

# Sync baseline config rows for every known check_name.
spark.sql(f"""
MERGE INTO {FULL_SCHEMA}.check_baseline_config c
USING (
    SELECT DISTINCT check_name
    FROM {FULL_SCHEMA}.check_registry
) s
ON c.check_name = s.check_name
WHEN NOT MATCHED THEN INSERT (check_name, baseline_mode, static_baseline_value, updated_at)
VALUES (s.check_name, 'MODEL', NULL, current_timestamp())
""")

# Normalize and validate baseline mode values.
spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_baseline_config
SET baseline_mode = UPPER(TRIM(COALESCE(baseline_mode, 'MODEL')))
""")

invalid_mode_count = spark.sql(f"""
SELECT COUNT(*) AS c
FROM {FULL_SCHEMA}.check_baseline_config
WHERE baseline_mode NOT IN ('MODEL', 'STATIC')
""").collect()[0]['c']
if invalid_mode_count > 0:
    raise RuntimeError("check_baseline_config has invalid baseline_mode values. Allowed: MODEL, STATIC")

duplicate_baseline_config_count = spark.sql(f"""
SELECT COUNT(*) AS c
FROM (
    SELECT check_name
    FROM {FULL_SCHEMA}.check_baseline_config
    GROUP BY check_name
    HAVING COUNT(*) > 1
) d
""").collect()[0]['c']
if duplicate_baseline_config_count > 0:
    duplicate_baseline_config_preview = spark.sql(f"""
    SELECT check_name, COUNT(*) AS duplicate_count
    FROM {FULL_SCHEMA}.check_baseline_config
    GROUP BY check_name
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC, check_name
    LIMIT 21
    """).toPandas()
    print("\nDuplicate check_name values detected in check_baseline_config:")
    print(duplicate_baseline_config_preview.head(20).to_string(index=False))
    if duplicate_baseline_config_count > 20:
        print(f"... and {duplicate_baseline_config_count - 20} more duplicate check_name row(s)")
    raise RuntimeError(
        "check_baseline_config contains duplicate check_name rows. "
        "Deduplicate baseline config before running jobs."
    )

missing_static_value_count = spark.sql(f"""
SELECT COUNT(*) AS c
FROM {FULL_SCHEMA}.check_baseline_config
WHERE baseline_mode = 'STATIC'
  AND static_baseline_value IS NULL
""").collect()[0]['c']
if missing_static_value_count > 0:
    raise RuntimeError(
        "check_baseline_config has STATIC checks with null static_baseline_value. "
        "Set a numeric static baseline in data_quality_update_checks_notebook."
    )

# Hard guardrail: fail setup if duplicate identity keys exist in any rows.
duplicate_key_count = spark.sql(f"""
SELECT COUNT(*) AS duplicate_key_count
FROM (
    SELECT check_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.check_registry
    GROUP BY check_name, workspace_id, dataset_id
    HAVING COUNT(*) > 1
) d
""").collect()[0]["duplicate_key_count"]

if duplicate_key_count > 0:
    duplicates_preview_df = spark.sql(f"""
    SELECT check_name, workspace_id, dataset_id, COUNT(*) AS duplicate_count
    FROM {FULL_SCHEMA}.check_registry
    GROUP BY check_name, workspace_id, dataset_id
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC, check_name, workspace_id, dataset_id
    LIMIT 21
    """).toPandas()

    active_duplicate_key_count = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_key_count
    FROM (
        SELECT check_name, workspace_id, dataset_id
        FROM {FULL_SCHEMA}.check_registry
        WHERE is_active = true
        GROUP BY check_name, workspace_id, dataset_id
        HAVING COUNT(*) > 1
    ) d
    """).collect()[0]["duplicate_key_count"]

    inactive_duplicate_key_count = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_key_count
    FROM (
        SELECT check_name, workspace_id, dataset_id
        FROM {FULL_SCHEMA}.check_registry
        WHERE is_active = false
        GROUP BY check_name, workspace_id, dataset_id
        HAVING COUNT(*) > 1
    ) d
    """).collect()[0]["duplicate_key_count"]

    print("\nDuplicate identity keys detected in check_registry:")
    print(duplicates_preview_df.head(20).to_string(index=False))
    if duplicate_key_count > 20:
        print(f"... and {duplicate_key_count - 20} more duplicate key(s)")
    print(f"Active duplicate keys: {active_duplicate_key_count}")
    print(f"Inactive duplicate keys: {inactive_duplicate_key_count}")
    raise RuntimeError(
        "check_registry contains duplicate (check_name, workspace_id, dataset_id) rows. "
        "Deduplicate rows before running add/validation notebooks."
    )
else:
    print("Uniqueness check passed: no duplicate identity keys found")

# Hard guardrail: IDs are required for stable identity and historical joins.
active_null_id_count = spark.sql(f"""
SELECT COUNT(*) AS missing_id_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
  AND (workspace_id IS NULL OR TRIM(workspace_id) = ''
   OR dataset_id IS NULL OR TRIM(dataset_id) = '')
""").collect()[0]["missing_id_count"]

if active_null_id_count > 0:
    active_null_id_preview_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND (workspace_id IS NULL OR TRIM(workspace_id) = ''
       OR dataset_id IS NULL OR TRIM(dataset_id) = '')
    ORDER BY check_name, model_name, workspace_id, dataset_id
    LIMIT 21
    """).toPandas()

    print("\nMissing workspace_id/dataset_id values detected in check_registry:")
    print(active_null_id_preview_df.head(20).to_string(index=False))
    if active_null_id_count > 20:
        print(f"... and {active_null_id_count - 20} more row(s)")
    raise RuntimeError(
        "check_registry has rows missing workspace_id or dataset_id. "
        "Update rows via data_quality_add_checks_notebook before running validation."
    )
else:
    print("ID check passed: all active rows include workspace_id and dataset_id")

active_invalid_threshold_count = spark.sql(f"""
SELECT COUNT(*) AS invalid_threshold_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
  AND (fail_delta_pct_threshold IS NULL OR fail_delta_pct_threshold < 0)
""").collect()[0]["invalid_threshold_count"]

if active_invalid_threshold_count > 0:
    active_invalid_threshold_preview_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id, fail_delta_pct_threshold
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND (fail_delta_pct_threshold IS NULL OR fail_delta_pct_threshold < 0)
    ORDER BY check_name, model_name, workspace_id, dataset_id
    LIMIT 21
    """).toPandas()

    print("\nInvalid fail_delta_pct_threshold values detected in check_registry:")
    print(active_invalid_threshold_preview_df.head(20).to_string(index=False))
    if active_invalid_threshold_count > 20:
        print(f"... and {active_invalid_threshold_count - 20} more row(s)")
    raise RuntimeError(
        "check_registry has active rows with null or negative fail_delta_pct_threshold values. "
        "Update rows via data_quality_update_checks_notebook before running validation."
    )
else:
    print("Threshold check passed: all active rows include a non-negative fail_delta_pct_threshold")

# Non-blocking visibility for legacy inactive rows that still have missing IDs.
inactive_null_id_count = spark.sql(f"""
SELECT COUNT(*) AS missing_id_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = false
  AND (workspace_id IS NULL OR TRIM(workspace_id) = ''
   OR dataset_id IS NULL OR TRIM(dataset_id) = '')
""").collect()[0]["missing_id_count"]

if inactive_null_id_count > 0:
    inactive_null_id_preview_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = false
      AND (workspace_id IS NULL OR TRIM(workspace_id) = ''
       OR dataset_id IS NULL OR TRIM(dataset_id) = '')
    ORDER BY check_name, model_name, workspace_id, dataset_id
    LIMIT 21
    """).toPandas()

    print("\nInactive rows with missing IDs were found (non-blocking):")
    print(inactive_null_id_preview_df.head(20).to_string(index=False))
    if inactive_null_id_count > 20:
        print(f"... and {inactive_null_id_count - 20} more row(s)")

# Hard guardrail: active rows must include workspace_name and dataset_name for DAX execution.
active_null_name_count = spark.sql(f"""
SELECT COUNT(*) AS missing_name_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
  AND (workspace_name IS NULL OR TRIM(workspace_name) = ''
   OR dataset_name IS NULL OR TRIM(dataset_name) = '')
""").collect()[0]["missing_name_count"]

if active_null_name_count > 0:
    active_null_name_preview_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_name, dataset_name
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND (workspace_name IS NULL OR TRIM(workspace_name) = ''
       OR dataset_name IS NULL OR TRIM(dataset_name) = '')
    ORDER BY check_name, model_name, workspace_name, dataset_name
    LIMIT 21
    """).toPandas()

    print("\nMissing workspace_name/dataset_name values detected in check_registry:")
    print(active_null_name_preview_df.head(20).to_string(index=False))
    if active_null_name_count > 20:
        print(f"... and {active_null_name_count - 20} more row(s)")
    raise RuntimeError(
        "check_registry has active rows missing workspace_name or dataset_name. "
        "Update rows via data_quality_add_checks_notebook before running validation."
    )
else:
    print("Name check passed: all active rows include workspace_name and dataset_name")

# Non-blocking visibility for inactive rows that still have missing names.
inactive_null_name_count = spark.sql(f"""
SELECT COUNT(*) AS missing_name_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = false
  AND (workspace_name IS NULL OR TRIM(workspace_name) = ''
   OR dataset_name IS NULL OR TRIM(dataset_name) = '')
""").collect()[0]["missing_name_count"]

if inactive_null_name_count > 0:
    inactive_null_name_preview_df = spark.sql(f"""
    SELECT check_name, model_name, workspace_name, dataset_name
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = false
      AND (workspace_name IS NULL OR TRIM(workspace_name) = ''
       OR dataset_name IS NULL OR TRIM(dataset_name) = '')
    ORDER BY check_name, model_name, workspace_name, dataset_name
    LIMIT 21
    """).toPandas()

    print("\nInactive rows with missing names were found (non-blocking):")
    print(inactive_null_name_preview_df.head(20).to_string(index=False))
    if inactive_null_name_count > 20:
        print(f"... and {inactive_null_name_count - 20} more row(s)")

# Hard guardrail: active rows must have valid run_frequency values.
active_invalid_frequency_count = spark.sql(f"""
SELECT COUNT(*) AS invalid_frequency_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
  AND UPPER(TRIM(COALESCE(run_frequency, ''))) NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
""").collect()[0]["invalid_frequency_count"]

if active_invalid_frequency_count > 0:
    active_invalid_frequency_preview_df = spark.sql(f"""
    SELECT check_name, model_name, run_frequency
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND UPPER(TRIM(COALESCE(run_frequency, ''))) NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
    ORDER BY check_name, model_name, run_frequency
    LIMIT 21
    """).toPandas()

    print("\nInvalid run_frequency values detected in active check_registry rows:")
    print(active_invalid_frequency_preview_df.head(20).to_string(index=False))
    if active_invalid_frequency_count > 20:
        print(f"... and {active_invalid_frequency_count - 20} more row(s)")
    raise RuntimeError(
        "check_registry has active rows with invalid run_frequency values. "
        "Allowed values are ONCE_PER_DAY or MULTIPLE_PER_DAY."
    )
else:
    print("Frequency check passed: all active rows use allowed run_frequency values")

# Non-blocking visibility for inactive rows with invalid run_frequency values.
inactive_invalid_frequency_count = spark.sql(f"""
SELECT COUNT(*) AS invalid_frequency_count
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = false
  AND UPPER(TRIM(COALESCE(run_frequency, ''))) NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
""").collect()[0]["invalid_frequency_count"]

if inactive_invalid_frequency_count > 0:
    inactive_invalid_frequency_preview_df = spark.sql(f"""
    SELECT check_name, model_name, run_frequency
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = false
      AND UPPER(TRIM(COALESCE(run_frequency, ''))) NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
    ORDER BY check_name, model_name, run_frequency
    LIMIT 21
    """).toPandas()

    print("\nInactive rows with invalid run_frequency values were found (non-blocking):")
    print(inactive_invalid_frequency_preview_df.head(20).to_string(index=False))
    if inactive_invalid_frequency_count > 20:
        print(f"... and {inactive_invalid_frequency_count - 20} more row(s)")

# Force policy: if a check has exactly one active row, that row must be baseline.
spark.sql(f"""
UPDATE {FULL_SCHEMA}.check_registry
SET is_baseline = true
WHERE is_active = true
  AND check_name IN (
      SELECT r.check_name
      FROM {FULL_SCHEMA}.check_registry r
      LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
        ON r.check_name = c.check_name
      WHERE r.is_active = true
        AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
      GROUP BY r.check_name
      HAVING COUNT(*) = 1
  )
""")

# Hard guardrail: MODEL checks must have exactly one active baseline row; STATIC checks can compare to numeric baseline.
active_baseline_issues_df = spark.sql(f"""
SELECT
    r.check_name,
    c.baseline_mode,
    COUNT(*) AS active_row_count,
    SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) AS active_baseline_count
FROM {FULL_SCHEMA}.check_registry r
LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
  ON r.check_name = c.check_name
WHERE r.is_active = true
GROUP BY r.check_name, c.baseline_mode
HAVING UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
   AND SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) <> 1
ORDER BY check_name
""").toPandas()

if len(active_baseline_issues_df) > 0:
    print("\nActive baseline invariant violations detected in check_registry:")
    print(active_baseline_issues_df.head(20).to_string(index=False))
    if len(active_baseline_issues_df) > 20:
        print(f"... and {len(active_baseline_issues_df) - 20} more check_name row(s)")
    raise RuntimeError(
        "Baseline invariant violation: each active MODEL check_name must have exactly one active baseline row. "
        "Fix baseline assignment/mode using data_quality_update_checks_notebook before running jobs."
    )
else:
    print("Baseline invariant check passed: each active MODEL check_name has exactly one active baseline")

# --- validation_results: written by the job, read-only for users ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.validation_results (
    run_id                     STRING    NOT NULL COMMENT 'UUID identifying this specific run',
    run_date                   DATE      NOT NULL,
    run_timestamp              TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    check_name                 STRING    NOT NULL,
    model_name                 STRING    NOT NULL,
    workspace_id               STRING    COMMENT 'Fabric workspace GUID copied from registry',
    dataset_id                 STRING    COMMENT 'Semantic model GUID copied from registry',
    workspace_name             STRING    COMMENT 'Workspace display name copied from registry',
    dataset_name               STRING    COMMENT 'Dataset display name copied from registry',
    result_value               DOUBLE    COMMENT 'Value returned by the DAX for this model',
    baseline_model             STRING    COMMENT 'The model flagged as is_baseline in check_registry - used as reference',
    baseline_value             DOUBLE    COMMENT 'Value returned by the DAX for the baseline model',
    absolute_delta             DOUBLE    COMMENT 'result_value minus baseline_value',
    delta_pct                  DOUBLE    COMMENT 'absolute_delta divided by baseline_value, as a percentage',
    fail_delta_pct_threshold   DOUBLE    COMMENT 'Percent delta threshold copied from check_registry for this row execution',
    status                     STRING    NOT NULL COMMENT 'PASS | FAIL | ERROR',
    error_message              STRING    COMMENT 'Populated only when status = ERROR'
 )
USING DELTA
PARTITIONED BY (run_date)
COMMENT 'Validation results written by the daily job - do not edit manually'
""")

# Ensure ID/name/threshold columns exist on older validation_results tables.
for add_col_sql in [
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ADD COLUMNS (workspace_id STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ADD COLUMNS (dataset_id STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ADD COLUMNS (workspace_name STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ADD COLUMNS (dataset_name STRING)",
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ADD COLUMNS (fail_delta_pct_threshold DOUBLE)",
]:
    try:
        spark.sql(add_col_sql)
        print(f"Applied schema update: {add_col_sql}")
    except Exception as e:
        if "already exists" not in str(e).lower():
            raise RuntimeError(
                f"Failed schema update for validation_results: {add_col_sql} -> {str(e)[:300]}"
            )

print(f"Tables ready in {FULL_SCHEMA}")
print("check_registry uniqueness validated on (check_name, workspace_id, dataset_id)")
print("workspace_id and dataset_id are enabled for stable model identity")
print("fail_delta_pct_threshold is stored per registry row and copied into validation_results")

## Verify

In [ ]:
from IPython.display import display

# Show both tables exist
tables = spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").toPandas()
display(tables)

required_tables = {"check_registry", "check_baseline_config", "validation_results"}

if "isTemporary" in tables.columns:
    temp_flag = tables["isTemporary"]
    if temp_flag.dtype == object:
        temp_flag = temp_flag.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
    else:
        temp_flag = temp_flag.fillna(False).astype(bool)
    tables_real = tables[~temp_flag]
else:
    tables_real = tables

name_col = None
if "tableName" in tables_real.columns:
    name_col = "tableName"
else:
    lowered = {str(c).strip().lower(): c for c in tables_real.columns}
    for candidate in ["tablename", "table_name", "name", "table"]:
        if candidate in lowered:
            name_col = lowered[candidate]
            break

if name_col is None:
    raise RuntimeError(
        "Could not locate table-name column in SHOW TABLES output. "
        f"Available columns: {', '.join(map(str, tables_real.columns.tolist()))}"
    )

found_tables = set(tables_real[name_col].astype(str).str.strip().str.lower().tolist())
missing_tables = sorted(required_tables - found_tables)

if missing_tables:
    raise RuntimeError(
        f"Missing required table(s) in {FULL_SCHEMA}: {', '.join(missing_tables)}"
    )

print(f"\n✓  Required tables created in {FULL_SCHEMA}")
print("   Next: go to data_quality_add_checks_notebook to add your checks, including fail_delta_pct_threshold per row")